# MovieLens 100k Data Exploration

This notebook explores the MovieLens 100k dataset and its IMDb metadata. It focuses on the structure of the data, dataset statistics, sparsity, and the key characteristics that make recommendation modeling challenging.

In [ ]:
from pathlib import Path
import csv
from collections import Counter, defaultdict

base = Path('ml-100k')

def read_ratings(path):
    ratings = []
    with open(path, 'r') as f:
        for row in csv.reader(f, delimiter='	'):
            ratings.append({
                'user_id': int(row[0]),
                'movie_id': int(row[1]),
                'rating': int(row[2]),
                'timestamp': int(row[3])
            })
    return ratings

def read_users(path):
    users = []
    with open(path, 'r') as f:
        for row in csv.reader(f, delimiter='|'):
            users.append({
                'user_id': int(row[0]),
                'age': int(row[1]),
                'gender': row[2],
                'occupation': row[3],
                'zip': row[4]
            })
    return users

def read_items(path):
    items = []
    with open(path, 'r', encoding='latin-1') as f:
        for row in csv.reader(f, delimiter='|'):
            row = row + [''] * (24 - len(row))
            items.append({
                'movie_id': int(row[0]),
                'title': row[1],
                'release_date': row[2],
                'video_release_date': row[3],
                'imdb_url': row[4],
                'genres': [int(x) for x in row[5:]]
            })
    return items

def read_genres(path):
    names = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('|')
            names.append(parts[0])
    return names

ratings = read_ratings(base / 'u.data')
users = read_users(base / 'u.user')
items = read_items(base / 'u.item')
genre_names = read_genres(base / 'u.genre')

print('loaded ratings:', len(ratings))
print('loaded users:', len(users))
print('loaded movies:', len(items))
print('loaded genres:', len(genre_names))

In [ ]:
rating_counts = Counter(r['rating'] for r in ratings)
movie_counts = Counter(r['movie_id'] for r in ratings)
user_counts = Counter(r['user_id'] for r in ratings)

genre_counts = Counter()
for item in items:
    for name, flag in zip(genre_names, item['genres']):
        if flag:
            genre_counts[name] += 1

num_users = len(users)
num_movies = len(items)
num_ratings = len(ratings)
sparsity = 1 - num_ratings / (num_users * num_movies)

print('### Dataset summary')
print('Users:', num_users)
print('Movies:', num_movies)
print('Ratings:', num_ratings)
print()
print('### Rating distribution')
for value in sorted(rating_counts):
    print(value, rating_counts[value])
print()
print('### Genre inventory')
for genre, count in genre_counts.most_common():
    print(f'{genre}: {count}')
print()
print('### Matrix sparsity')
print(f'sparsity = {sparsity:.6f} ({sparsity * 100:.2f}% empty)')
print()
print('### Interaction density')
print('avg ratings per user:', num_ratings / num_users)
print('avg ratings per movie:', num_ratings / num_movies)
print()
print('### Rating coverage extremes')
print('min ratings per user:', min(user_counts.values()))
print('max ratings per user:', max(user_counts.values()))
print('min ratings per movie:', min(movie_counts.values()))
print('max ratings per movie:', max(movie_counts.values()))
print()
print('### Top 10 most rated movies')
for movie_id, count in movie_counts.most_common(10):
    title = next((m['title'] for m in items if m['movie_id'] == movie_id), 'unknown')
    print(movie_id, title, count)

## Feature Inventory

The MovieLens 100k dataset contains the following primary feature groups:

- User features: age, gender, occupation, zip code
- Item features: movie title, release date, IMDb URL, genre flags
- Interaction features: rating score, timestamp

This mix supports collaborative filtering, content filtering, and hybrid recommendation approaches.

In [ ]:
print('### Sample users')
for user in users[:5]:
    print(user)
print()
print('### Sample movies')
for item in items[:5]:
    print({
        'movie_id': item['movie_id'],
        'title': item['title'],
        'release_date': item['release_date'],
        'genres': [name for name, flag in zip(genre_names, item['genres']) if flag],
    })
print()
print('### Sample ratings')
for r in ratings[:5]:
    print(r)

## Why recommendations are difficult

The dataset shows several key challenges for recommendation systems:

- High sparsity: more than 93% of the user-item matrix is empty, meaning most users have not rated most movies.
- Long tail: a small number of movies receive many ratings while many movies receive very few ratings.
- Cold start: new users or movies have little or no interaction history.
- Content diversity: items span many genres, and users may prefer a mix of categories.

Understanding these data characteristics is essential before choosing modeling techniques.